<div align="center">
  <img src="https://raw.githubusercontent.com/NaumanHSA/neurosurfer/main/docs/assets/banner/neurosurfer-banner-light.png" alt="Neurosurfer" width="45%"/>
</div>

<br/>

# 06 — The Architect

Every previous notebook had **you** design the workflow. You picked the nodes, wrote the prompts,
wired the `depends_on` edges. This one hands that job to an agent.

The **Architect** takes a plain-English intent and produces a registered, runnable
[WorkflowPackage](03_graph_agents.ipynb) — it decides the steps, finds the tools each one needs,
writes the prompts, wires the graph, **runs what it built**, judges the output against criteria it
derived itself, and repairs anything that failed. If it can't do the job it refuses and tells you
exactly what's missing, rather than shipping a workflow that pretends.

You'll learn:

1. **The two-agent split** — why planning and building are separate calls, and why that is the
   single biggest factor in whether this works on a small model.
2. **The plan** — `plan_and_resolve`, and the capability ladder that runs *in code*.
3. **The build** — `ArchitectAgent.build`, watched step by step.
4. **The gate** — validation, and the rule that stops a workflow from accepting a parameter it
   ignores.
5. **The proof** — `derive_acceptance` + `verify_workflow`: the closed loop that runs and judges
   the workflow before it registers.
6. **The refusal** — `WorkflowInfeasible`, and why a clear blocker beats a plausible fake.

> **Runtime:** ~3–10 minutes on a local 9B model — it varies a lot, because how long §5 takes
> depends on how much the Architect has to repair. Every section prints as it goes, so you can
> watch rather than wait.

---

## 1. Setup

Same boilerplate as the earlier notebooks. The Architect is pure Python — no optional extras
needed beyond what you already have.

The validated local choice is **`qwen/qwen3.5-9b`** in LM Studio, with the local server on
(port 1234). A 9B model is a deliberate test: the Architect is designed so that *the framework*
carries the correctness, not the model's memory. If it works here, a bigger model is strictly
easier.

In [2]:
%load_ext autoreload
%autoreload 2

import sys, os, json, textwrap
from pathlib import Path

# Point Python at the repo root when running from tutorials/
NB_DIR    = Path(os.getcwd())
REPO_ROOT = NB_DIR.parent
if str(REPO_ROOT) not in sys.path:
    sys.path.insert(0, str(REPO_ROOT))

import neurosurfer
print(f"neurosurfer {neurosurfer.__version__}")

# Everything the Architect writes lands here — staging while it builds, then the
# registry. `tutorials/tmp/` is gitignored, and it is worth opening afterwards:
# the graph.yaml in there is the actual deliverable.
WORK     = NB_DIR / "tmp" / "architect"
STAGING  = WORK / "staging"
REGISTRY = WORK / "registry"
WORK.mkdir(parents=True, exist_ok=True)
print(f"working dir: {WORK}")

version,1.0.0 | python 3.11.15
os,Linux 6.17.0-41-generic (x86_64)
torch,2.11.0+cu128 CUDA: yes (12.8)
mps,no (built: False)
transformers,4.51.3 sentEmb 5.5.0
accelerate,1.13.0 bnb -
gpu,NVIDIA GeForce RTX 5080


neurosurfer 1.0.0
working dir: /home/nomi/workspace/neurosurfer/tutorials/tmp/architect


In [3]:
# ── LM Studio connection ──────────────────────────────────────────────────────
LM_STUDIO_URL   = "http://localhost:1234/v1"
LM_STUDIO_MODEL = "qwen/qwen3.5-9b"
CONTEXT_WINDOW  = 32_768

from neurosurfer.llm.providers.openai import OpenAICompatProvider

provider = OpenAICompatProvider(
    model          = LM_STUDIO_MODEL,
    base_url       = LM_STUDIO_URL,
    api_key        = "lm-studio",
    context_window = CONTEXT_WINDOW,
)

# For a hosted model instead, swap the two lines above for:
#   from neurosurfer.llm.providers.openai import OpenAIProvider
#   provider = OpenAIProvider(model="gpt-5-mini", api_key=os.environ["OPENAI_API_KEY"])

print(f"provider ready: {LM_STUDIO_MODEL}")

provider ready: qwen/qwen3.5-9b


In [4]:
# A tiny progress printer. Every Architect entry point takes `notify=`, and it is
# the whole reason this notebook is watchable rather than a 6-minute blank cell.
def say(msg: str) -> None:
    print(f"  · {msg}")

---

## 2. What the Architect actually is

Two agents, not one — and that split is the whole design.

```
      your intent
           │
           ▼
   ┌───────────────┐   ONE structured call. No tools, no loop.
   │    PLANNER    │   "What are the steps, and which reach outside the model?"
   └───────┬───────┘
           │  WorkflowPlan
           ▼
   ┌───────────────┐   Runs IN CODE, not by asking the model:
   │   CAPABILITY  │   catalog search → MCP registry → author a tool → block
   │     LADDER    │   Writes the answer back onto each step.
   └───────┬───────┘
           │  plan, with tools already attached
           ▼
   ┌───────────────┐   A ReAct agent with a 17-tool belt:
   │    BUILDER    │   add_node · validate_workflow · test_workflow · register
   └───────┬───────┘
           │
     ┌─────┴──────┬─────────────┐
     ▼            ▼             ▼
  registered   blocked      gave up
                            (with the last report)
```

**Why split it?** Holding requirements, design, tool-sourcing, wiring, validation *and*
verification in one stream is exactly what a small model drops things from. Planning is a single
question with a single schema-shaped answer — the form a weak model is most reliable at. The
builder then gets told *which tool to use* rather than being asked to remember one.

The source says it plainly:

> *"By the time the builder sees the plan, a step needing a file has `read_file` attached to it.
> The builder is told which tool to use rather than asked to remember one — which is the single
> biggest difference between this working on gpt-4o-mini and not."*
> — `neurosurfer/architect/planner.py`

---

## 3. The plan, on its own

You can run the planner without building anything. This is one model call and it is the cheapest
possible way to see what the Architect *thinks* your request is.

Our intent has a deliberate mix: one step that must reach outside the model, and two that must not.

In [ ]:
INTENT = (
    "Read a text file of customer feedback, pull out the recurring complaints, "
    "and write a short summary for the support lead."
)

from neurosurfer.architect.planner import plan_and_resolve

plan = await plan_and_resolve(provider, INTENT, notify=say)

if not plan.steps:
    # Worth seeing rather than hiding: a small model does occasionally return a
    # plan with no steps. `build()` treats planning as an advantage, not a gate —
    # it notices this and builds without a plan — so §5 works either way.
    print("\n!! the planner returned no steps this run. Re-run this cell to try again;\n"
          "   §5 does not depend on it.")

print(f"\nname:        {plan.name}")
print(f"description: {plan.description}")
print(f"inputs:      {[i.name for i in plan.inputs]}")
print(f"outputs:     {plan.outputs}")

> /tmp/ipykernel_346134/3161103873.py(8)<module>()
      6 )
      7 
----> 8 from neurosurfer.architect.planner import plan_and_resolve
      9 
     10 plan = await plan_and_resolve(provider, INTENT, notify=say)



### Reading the plan

The field that matters on every step is **`is_external`** — *does this reach outside the model?*

Reading a file, fetching a URL, calling an API, sending a message: none of that can be done by an
LLM step, however well you word it. A `base` node asked to "read the contents of {path}" does not
read anything — it **invents a plausible file**. That single failure mode is what the whole
capability ladder exists to prevent.

In [ ]:
for s in plan.steps:
    mark = "EXTERNAL" if s.is_external else "internal"
    print(f"[{mark:8}] {s.id}  ({s.kind_hint})")
    print(f"             {s.intent}")
    if s.depends_on:
        print(f"             after: {', '.join(s.depends_on)}")
    if s.is_external:
        print(f"             needs: {s.needed_capability}")
        res = s.resolution or {}
        print(f"             ladder -> status={res.get('status')!r}  tool={res.get('tool')!r}")
    print()

Notice the last two lines on the external step. That `resolution` was **not** produced by the
model — `resolve_plan` walked the ladder in plain Python and wrote the answer back onto the step:

1. search the registered tool catalog,
2. then the MCP registry,
3. then decide: assign it, author a new tool, install a server, or block.

If nothing anywhere provides the capability, the build stops *here* — before a single node exists,
before any prompt is written, and before you have spent a build's worth of tokens on a workflow
that could never have run.

---

## 4. The builder's contract

Before we run it: the builder is a ReAct agent, and its system prompt gives it a toolbelt of 17
tools plus one hard rule about when it is allowed to stop.

| Tool | What it does |
|---|---|
| `set_workflow`, `set_outputs` | name, description, declared inputs, result nodes |
| `add_node`, `update_node`, `remove_node` | build the graph one node at a time |
| `validate_workflow` | run the full rule table — **errors block** |
| `test_workflow` | derive criteria, actually run it, judge the output |
| `find_capability` | search catalog + MCP registry (never guess a tool name) |
| `author_tool`, `install_mcp_server` | acquire a capability that is missing |
| `register_workflow`, `declare_blocked` | the only two ways to finish |

> **"You are finished ONLY after `register_workflow` succeeds or `declare_blocked` is called.
> A turn with only text and no tool call before then is a mistake."**

That last clause is load-bearing. Small models like to end a turn narrating — *"now let me add the
summarise node…"* — which an agent loop reads as a final answer. The Architect detects that and
**nudges** the same conversation back to work rather than restarting it.

---

## 5. Build it

This is the long cell — anywhere from **90 seconds to 8 minutes** on a 9B model. Watch the `·`
lines: you are seeing the plan, then nodes appearing, then validation, then verification.

`build()` plans internally, so it re-runs §3's step for itself. You *can* skip that by passing
`plan=plan` — which is how a plan reviewed in one request gets built by the next one without being
re-invented in between — but note the difference in tolerance:

| | empty / failed plan |
|---|---|
| `build(intent)` | notices, says *"building without a plan"*, carries on |
| `build(intent, plan=p)` | raises `ValueError` — you handed it the plan, so it holds you to it |

Planning is an advantage, not a gate. We take the tolerant path here so this notebook runs even
when a 9B model has an off moment.

In [ ]:
import time
from neurosurfer.architect import ArchitectAgent
from neurosurfer.graph.workflow.registry import WorkflowRegistry

agent = ArchitectAgent(
    provider,
    registry     = WorkflowRegistry(workflows_dir=REGISTRY),
    staging_root = STAGING,
    notify       = say,
    max_turns    = 30,
)

t0 = time.time()
package_path = await agent.build(INTENT)
print(f"\nregistered in {time.time() - t0:.0f}s")
print(package_path)

### What those lines meant

A clean run — the plan was good, so nothing needed repairing:

```
· planned 2 step(s), 1 needing an external capability
·   load_feedback: read a text file from disk → have
· node added: load_feedback (tool) [1/2 steps]
· node added: extract_complaints_summary (base) [2/2 steps]
· validate: ok
· deriving acceptance criteria + test inputs
· verification PASSED (1 graph run; 1 this build)
· REVIEW OK — ...
· Workflow 'summarize_customer_feedback' registered at ...
```

Two things to notice. **`read a text file from disk → have`** is the ladder finding `read_file` in
the catalog, in code, before any node existed. And **`[1/2 steps]`** is the builder tracking the
plan: every planned step must be built or explicitly dropped, so it cannot quietly lose one.

When verification *doesn't* pass first time you get the repair loop instead:

```
· verification FAILED (1 graph run; 1 this build)
· verification FAILED (1 graph run; 2 this build)
· verification PASSED (1 graph run; 3 this build)
```

Each failure is followed by a fix and another real run. The builder is told to **fix the smallest
thing first** — sharpen the failing node's prompt, correct its wiring, swap a tool — and explicitly
*not* to escalate a working router into a loop to patch what is really a prompt bug. The counter
reads `N graph run; M this build`: runs inside *this* verification, then the build's running total.

> This is the expensive half of the Architect, and the part most worth watching. On harder intents
> it is where the time goes.

---

## 6. Read what it wrote

The output is not a black box — it is an ordinary WorkflowPackage on disk. Same format you would
have written by hand in notebook 03.

In [ ]:
from neurosurfer.graph.workflow.package import load_package

pkg = load_package(Path(package_path))

print(f"name:    {pkg.name}")
print(f"inputs:  {[i.name for i in pkg.graph.inputs]}")
print(f"outputs: {pkg.graph.outputs}\n")

for n in pkg.graph.nodes:
    after = f"  after: {', '.join(n.depends_on)}" if n.depends_on else ""
    print(f"  {n.id}  ({n.kind}){after}")
    if n.tools:
        print(f"      tools: {n.tools}")

In [ ]:
# The graph.yaml itself. This is the deliverable — commit it, edit it, run it in CI.
print((Path(package_path) / "graph.yaml").read_text()[:2500])

### The prompts are the interesting part

Look at what it wrote into a node's `purpose` / `goal`, and specifically at the `{placeholders}`.
A node receives **only** what its own text names plus its declared dependencies — there is no
ambient block of every graph input. So an input that appears in no prompt is an input that reaches
no step, which is exactly what §7 is about.

In [ ]:
first = pkg.graph.nodes[0]
for field in ("instructions", "purpose", "goal", "expected_result"):
    val = getattr(first, field, None)
    if val:
        print(f"── {field} ──")
        print(textwrap.indent(textwrap.fill(str(val), 92), "   "), "\n")

---

## 7. The gate: validation

Nothing registers unless `validate_package` passes. The rules are a table, not a wall of `if`s, and
they split into two severities that mean very different things:

- **ERROR** — the workflow will not run, or will not run correctly. **Blocks registration.**
- **WARNING** — it runs, but something is probably not what was meant.

Our built package passes, obviously. The interesting demo is a graph that *looks* fine and isn't.

In [ ]:
from neurosurfer.graph.workflow.validation import validate_package

report = validate_package(pkg)
print(f"ok: {report.ok}   errors: {len(report.errors)}   warnings: {len(report.warnings)}")
if report.issues:
    print(report.summary())

### The rule that catches a workflow which ignores you

Here is a graph that loads, runs green, and is wrong: it **declares** an input and no step ever
names it. A caller passes their article; the model never sees it; the answer is confident and
unrelated.

This used to be a warning, so the workflow registered and the backstop was a human noticing the
answer had nothing to do with what they passed. That is not a backstop a workflow the Architect
builds and verifies *on its own* ever gets — so it blocks now.

In [ ]:
from neurosurfer.graph.engine.schema import Graph, GraphInput, GraphNode
from neurosurfer.graph.workflow.package import WorkflowPackage
from neurosurfer.graph.workflow.schema import WorkflowManifest

def check(node_instructions: str):
    # A one-node graph that declares an `article` input.
    graph = Graph(
        name   = "demo",
        nodes  = [GraphNode(id="a", kind="base", instructions=node_instructions)],
        outputs= ["a"],
        inputs = [GraphInput(name="article", type="string")],
    )
    pkg = WorkflowPackage(manifest=WorkflowManifest(name="demo"), graph=graph, path=Path("."))
    return validate_package(pkg)

# ── the broken one: `article` is declared, and named nowhere ──────────────────
bad = check("Write a summary.")
print(f"registers? {bad.ok}")
for e in bad.errors:
    print(f"  ERROR  {e.message}")
    print(f"         -> {e.suggestion}")

In [ ]:
# ── the fix: name it, and it is read ─────────────────────────────────────────
good = check("Summarise {article}.")
print(f"registers? {good.ok}   (issues: {len(good.issues)})")

The rule counts **every** way a value can be read, not just prompt placeholders — a `map`'s `over`
expression, a `when` guard, `tool_args`, an output node's `value`, a code node's parameter names,
and nodes nested inside container bodies. A rule that only looked at `instructions` would report a
perfectly good fan-out as ignoring its collection.

And it **downgrades itself to a warning where it cannot see**: a `tool` node's arguments live in a
registered schema, and a callable may not import or inspect. Either hides the reads that would
clear the input — and refusing to run over a fact that was never established is worse than the gap.

---

## 8. The proof: verification

Validation asks *"is this well-formed?"*. Verification asks the much harder question:
**"does it actually do what was asked?"** — and answers it by running the thing.

Two calls, and you can drive them yourself:

1. **`derive_acceptance`** — one model call turns the intent + the graph's declared inputs into
   2–6 explicit success criteria, concrete test inputs, and — when the workflow reads a file or a
   directory — a **fixture** that *creates* it.
2. **`verify_workflow`** — runs the staged package on those inputs in a throwaway sandbox, then
   scores each criterion with a judge, **fail-closed** (a criterion the judge doesn't rule on
   counts as failed).

That fixture step is why this is real rather than theatre: a workflow that reads a file cannot be
tested against a *sentence*. A source path with nothing behind it is a hard failure naming the
fixture it needs — not a placeholder string handed to `read_file` so it can fail as "no such file".

> **Heads up:** §5 already verified this workflow successfully, using a rig the agent derived for
> itself. Below we derive a **fresh** rig — and a 9B model sometimes writes a fixture script that
> doesn't parse. If that happens you'll see a FAILED report, and the next section explains why that
> is the system working. Re-running the cell usually produces a script that runs.

In [ ]:
from neurosurfer.architect.agent import derive_acceptance, verify_workflow

declared = [i.model_dump() for i in pkg.graph.inputs]
graph_yaml = (Path(package_path) / "graph.yaml").read_text()

acceptance = await derive_acceptance(provider, INTENT, graph_yaml, declared_inputs=declared)

print("test inputs:", json.dumps(acceptance.test_inputs, indent=2)[:500], "\n")

print("criteria the workflow must satisfy:")
for c in acceptance.criteria:
    print(f"  [{c.id}] {c.description}")

if acceptance.fixtures:
    # A single Fixture: a setup script, and the paths it promises to create.
    print("\nfixture — this workflow reads from disk, so real files get made first:")
    print(f"  creates: {', '.join(acceptance.fixtures.creates)}")

In [ ]:
# Actually run it and judge the result. ~1-2 minutes.
report = await verify_workflow(
    provider,
    intent       = INTENT,
    package_dir  = Path(package_path),
    plan         = acceptance,
    declared_inputs = declared,
)

print(f"PASSED: {report.passed}\n")
print(report.render() if hasattr(report, "render") else report)

### Reading the report — and the distinction that matters

Two very different failures wear the same word, and the report never confuses them:

- **The workflow is wrong.** The run completed and a criterion failed. You get per-criterion
  verdicts plus design suggestions, and the fix belongs in the graph.
- **The test rig is wrong.** The fixture script didn't run, so the workflow never got a fair
  attempt. Then the diagnosis says so in as many words:

  > *"Could not set up the test fixtures — the fixture setup script failed … **This is the test
  > rig, NOT the workflow — do not change the graph, and do not add nodes to create test files.**"*

That last sentence is there because of a real failure mode: told only that "the test file was
missing", the agent started adding `setup_fixtures` / `cleanup_fixtures` **function nodes to the
workflow** — permanently deforming a correct design to satisfy a broken harness. Naming which side
is at fault is what stops that.

And the scoring is **fail-closed**: a criterion the judge doesn't rule on counts as failed. A
verification that cannot reach a verdict is a verification that did not pass — never a pass by
default.

---

## 9. When it refuses

The most important behaviour in the whole subsystem is the one that produces *nothing*.

Ask for something that needs credentials or resources you never supplied, and the Architect calls
`declare_blocked` — which surfaces as `WorkflowInfeasible` carrying a checklist of exactly what is
missing. A clear blocker beats a workflow that pretends.

Note what is **not** a blocker: needing to analyse, summarise, classify or write is never a reason
to refuse. That is what a `base` node is for. The refusal is reserved for capabilities that
genuinely are not available.

In [ ]:
from neurosurfer.architect import WorkflowInfeasible

blocked_agent = ArchitectAgent(
    provider,
    registry     = WorkflowRegistry(workflows_dir=REGISTRY),
    staging_root = STAGING / "blocked",
    notify       = say,
    max_turns    = 20,
)

try:
    await blocked_agent.build(
        "Build a workflow that logs into my company's production Oracle database "
        "and deletes duplicate customer rows."
    )
    print("!! it built something — that is a finding, not a pass")
except WorkflowInfeasible as e:
    print("\nBLOCKED, as it should be:\n")
    print(textwrap.indent(textwrap.fill(str(e), 92), "  "))

### What a good refusal looks like

Notice how far it got before refusing. It planned three steps, resolved two of them to the `sql`
tool, built all three nodes — and *then* blocked, because validation showed the capability it had
been given could not do the job:

> *"the `sql` tool in the catalog is **read-only** and cannot perform DELETE operations … The
> available capabilities are: test_connection, list_tables, table_schema, query. To build this
> workflow, either (1) `author_tool` to create a Python tool that can execute write operations, or
> (2) install an MCP server that provides read-write database access."*

That is the shape to want: it names the **specific** gap (write access, not "database access"),
lists what it *does* have, and gives two concrete routes forward. Compare it to the alternative —
a registered workflow with a `remove_duplicates` node that quietly does nothing, and a green run
telling you your duplicates are gone.

Also worth noting: it did **not** block on the analysis steps. Needing to summarise, classify or
write is never a reason to refuse — that is what a `base` node is for. Only a genuinely absent
capability blocks.

---

## 10. Where this runs from

Two doors into the same Architect:

| Door | How |
|---|---|
| **Python** | `ArchitectAgent(provider).build(intent)` — what this notebook used |
| **HTTP gateway** | `POST /v1/architect/builds`, then stream `GET /v1/architect/builds/{id}/events` for the step log and staged-graph snapshots |

> ⚠️ One inconsistency worth knowing: the CLI's `neurosurfer workflow create` routes to
> `ArchitectBuilder`, an **older fixed 8-node pipeline** (`discover → clarify → decompose →
> design_nodes → critique → tool_design → write_nodes → assemble`) that still ships in
> `neurosurfer/architect/package/`. `ArchitectAgent` — the ReAct agent this notebook drove — is the
> current path, and it is what the gateway serves. Prefer the Python or HTTP route until the CLI is
> moved over.

Once registered, a workflow is just a package. Run it like any other:

```python
from neurosurfer.graph.workflow.runner import WorkflowRunner
result = WorkflowRunner(provider).run(pkg, {"feedback_file": "feedback.txt"})
```

---

## Summary

You handed an agent a sentence and got back a runnable, validated, **tested** workflow package.

| Stage | Call | What it guarantees |
|---|---|---|
| Plan | `plan_and_resolve` | one structured answer — the form a weak model is reliable at |
| Ladder | `resolve_capability` | tools chosen **in code**, not remembered by the model |
| Build | `ArchitectAgent.build` | one node at a time, warnings read after every call |
| Gate | `validate_package` | errors block; an ignored parameter is now one of them |
| Proof | `derive_acceptance` + `verify_workflow` | it ran, on a real fixture, judged fail-closed |
| Refusal | `WorkflowInfeasible` | a checklist, not a workflow that pretends |

### The three ideas worth taking away

1. **Split the hard question from the long one.** Planning is one schema-shaped call; building is a
   loop. Merging them is what makes small models drop requirements.

2. **Do the deterministic part deterministically.** The capability ladder is plain Python. Asking a
   model to remember tool names is the most reliable way to get invented ones.

3. **A workflow that runs green is not a workflow that works.** Structure validation and behaviour
   verification are different questions, and the Architect is only trustworthy because it asks
   both — then repairs what it broke and asks again.

### Where to go next

- **[03 — Graph Agents](03_graph_agents.ipynb)** — the format the Architect emits, by hand.
- **[05 — Capstone](05_capstone_insight_engine.ipynb)** — a seven-node graph a human designed.
- Open `tutorials/tmp/architect/registry/` and read the `graph.yaml` it wrote. Editing it by hand
  is the fastest way to understand what it decided, and why.

In [ ]:
print('Plan a workflow for this request:\n\nBuild a workflow that takes a short article text as input, summarises it in 3 sentences, and then writes a catchy title for the summary.\n\nkind_hint must be one of base, tool, react, router, loop, map. Mark every step that reaches outside the model as is_external.')

In [2]:
print('{\n  "name": "summarize_and_title_article",\n  "description": "Summarizes an article into three sentences and generates a catchy title for the summary.",\n  "inputs": [\n    {\n      "name": "article_text",\n      "type": "string",\n      "required": true,\n      "description": "The full text of the short article to process."\n    }\n  ],\n  "steps": [\n    {\n      "id": "summarize_article",\n      "intent": "Summarize the input article into exactly three sentences.",\n      "kind_hint": "base",\n      "depends_on": [],\n      "produces": "summary_text",\n      "is_external": false,\n      "secrets": []\n    },\n    {\n      "id": "generate_title",\n      "intent": "Create a catchy title for the generated summary text.",\n      "kind_hint": "base",\n      "depends_on": [\n        "summarize_article"\n      ],\n      "produces": "title",\n      "is_external": false,\n      "secrets": []\n    }\n  ],\n  "outputs": [\n    "generate_title"\n  ],\n  "open_questions": []\n}')

{
  "name": "summarize_and_title_article",
  "description": "Summarizes an article into three sentences and generates a catchy title for the summary.",
  "inputs": [
    {
      "name": "article_text",
      "type": "string",
      "required": true,
      "description": "The full text of the short article to process."
    }
  ],
  "steps": [
    {
      "id": "summarize_article",
      "intent": "Summarize the input article into exactly three sentences.",
      "kind_hint": "base",
      "depends_on": [],
      "produces": "summary_text",
      "is_external": false,
      "secrets": []
    },
    {
      "id": "generate_title",
      "intent": "Create a catchy title for the generated summary text.",
      "kind_hint": "base",
      "depends_on": [
        "summarize_article"
      ],
      "produces": "title",
      "is_external": false,
      "secrets": []
    }
  ],
  "outputs": [
    "generate_title"
  ],
  "open_questions": []
}


In [4]:
print("""
'You are the Neurosurfer Architect — an agent that designs and builds runnable\nworkflow packages from a user\'s plain-English intent.\n\n# Operating procedure\n1. BUILD. `set_workflow` first (name, description, declared inputs), then\n   `add_node` one node at a time — read every warning and fix it.\n   - **With a plan** (usually — it is in your first message): build it. One node\n     per step, keeping the step ids. The tools are already chosen for you; use\n     exactly those. If a step\'s id differs from your node\'s, pass `plan_step_id`.\n     Every planned step must be built or explicitly `drop_plan_step`ped.\n   - **Without a plan**: design as you go. One node per distinct capability or\n     decision; let the work set the size, not a target count. Map each node to a\n     clause the user actually asked for — do NOT add validation, formatting or\n     "nice-to-have" steps they did not request.\n2. GROUND anything the plan didn\'t. For each node ask: *does this reach outside\n   the model?* Reading a file, fetching a URL, calling an API, touching an inbox,\n   sending a message, running a command — none of that can be done by an LLM step,\n   however well you word it. Such a node MUST have a tool; `find_capability` finds\n   it. See "Capability grounding" below; the validator enforces it.\n3. VERIFY structure: `validate_workflow`, fix every reported issue, repeat until\n   VALID.\n4. PROVE it works: `test_workflow` actually runs your workflow on realistic\n   derived inputs and judges the outputs against the user\'s intent. If it FAILS,\n   read the diagnosis and FIX THE SMALLEST THING FIRST: sharpen the failing\n   node\'s `purpose`/`goal`/`expected_result`, correct its `depends_on` wiring, or\n   swap a tool. Only add a new node or a control-flow construct if the intent\n   truly needs a step that is missing — do NOT escalate a working `router` into a\n   `loop` (or bolt on extra nodes) to patch what is really a prompt bug. Never\n   leave a failing verification unaddressed.\n5. FINISH: declare the result node(s) with `set_outputs` (never `update_node`),\n   then `register_workflow` once valid (and tested), and stop with a 2–3\n   sentence summary of what the workflow does and its inputs. If the request is\n   impossible as described (needs credentials/resources the user didn\'t provide,\n   or an unsafe capability), call `declare_blocked` with a precise reason instead.\n\nWhen unsure how a construct works, use `describe_capability` — the node kinds,\ntheir required fields and worked shapes are yours already, above. `neurosurfer_docs`\nis for project background (configuration, the CLI, providers), NOT for authoring:\nthose docs are written for people setting neurosurfer up and lag the code, so they\nanswer build questions with setup guides. `web_search` only for domain research,\nnever for neurosurfer questions.\n\n# CRITICAL — keep going until done\n- Drive the whole build yourself in ONE session. After every tool result,\n  IMMEDIATELY make the next tool call. Do NOT stop to narrate progress or ask the\n  user anything — the intent is already given.\n- You are finished ONLY after `register_workflow` succeeds or `declare_blocked` is\n  called. A turn with only text and no tool call before then is a mistake.\n\n# Node-authoring rules\n- Every base/react node needs a clear `purpose` (its system prompt), a `goal`, and\n  an `expected_result`. Reference declared workflow inputs as `{input_name}`.\n- WIRING IS MANDATORY: data flows ONLY along `depends_on` edges. Any node that\n  uses another node\'s result MUST list that node in `depends_on` (e.g. a\n  title-writing step that uses the summary MUST depend on the summarise step).\n  A multi-node workflow with no `depends_on` edges is wrong — it is a bag of\n  parallel nodes, not a pipeline.\n- NO ORPHAN NODES: every node\'s output must be consumed — either declared in\n  `set_outputs` or listed in a downstream node\'s `depends_on`. If nothing uses a\n  node\'s result, delete the node; it is dead weight, not a feature.\n- Guards over LLM text: prefer `contains(lower(nodes.x), \'label\')` — never exact\n  equality against raw model output.\n- Router case targets (and `on_error` targets) must list the router/node in their\n  `depends_on`.\n- Loops always need `max_iterations`; maps always need `over`.\n\n# Capability grounding — THE most common way a build fails\nAn LLM node only ever sees its prompt. Wording a `base` node as "Read the contents\nof {file_path}" does not make it read anything — it makes it INVENT a plausible\nfile. The plan has usually decided this for you; where it hasn\'t, ask per node:\n**does this reach outside the model?**\n- NO (write, summarise, classify, rewrite, judge, reason over text it was given)\n  → `base`, no tools.\n- YES, one definite call → `tool`. YES, several steps or a choice between them →\n  `react` WITH tools. A tool-less `react` is refused by the validator and engine.\n\n**NEVER guess a tool name.** `find_capability("<the capability in plain English>")`\nsearches the catalog then the MCP registry and says what each option costs. Then\ntake exactly one of these, never papering over it:\n  1. **assign a tool it found** — `update_node` with the right kind and `tools`;\n  2. `author_tool` — buildable from plain Python plus a value the user can paste,\n     sandbox-tested and human-approved. Reach for this as readily as for an\n     install; it is often quicker and always safer;\n  3. `install_mcp_server` — when the capability needs a vendor account, a browser\n     sign-in or an SDK that a Python file cannot reproduce;\n  4. `declare_blocked` — name the server and the exact variables it wants, so the\n     user gets a checklist rather than an apology. A clear blocker beats a workflow\n     that pretends. But NEVER block because a step needs analysis, summarising,\n     classifying or writing — that is what a `base` node is for;\n  5. `acknowledge_capability` — only when the check is wrong and the step needs no\n     tool (e.g. its data already arrives from an upstream node).\n\n## Acquiring a capability — the loop, in order\nA step marked INSTALLABLE is **not** a blocker. Getting it is your job, not the\nuser\'s, and there are always TWO ways to get it:\n\n  1. `find_capability` — read the shortlist. Prefer a server whose description\n     names the thing you actually need over one that merely shares a word. **A\n     shortlist of servers that do not match is the same as an empty one** — being\n     offered a flood-data server for "read environment variables" means nothing\n     here provides it, not that you should install a flood-data server.\n  2. **Decide which route the capability needs.**\n     - `author_tool` when it is plain Python plus a value the user can paste: a\n       database query, a file conversion, a REST call, a calculation. Three or\n       four small tools usually cover a whole integration. This route is also\n       safer — your tool is sandbox-tested and approved here, where an install\n       runs someone else\'s code.\n     - `install_mcp_server` when it needs a vendor account, a browser sign-in, an\n       OAuth grant, or a proprietary SDK — things a Python file cannot reproduce.\n  3. **Check what a server exposes before installing it.** Where the shortlist\n     shows a server\'s tools, read them; otherwise install and then\n     `list_mcp_tools`. A server whose description sounded right and whose tools\n     are wrong is the common failure, and it costs a whole build to find late.\n  4. `install_mcp_server` **asks the user** before running anything; you do not\n     need permission to try. Credentials already configured here are filled in for\n     you, so supply only values the user gave you in this conversation, and never\n     invent one.\n  5. If an install is declined or needs a value nobody gave you, do not stop at the\n     next server — ask whether you could simply **write** the tool.\n  6. Acquiring settles the steps that were waiting on it. **Then look again** at\n     what is still unresolved and repeat, until every step has something real\n     behind it. Only then start writing nodes.\n\n`declare_blocked` is for what neither route reaches: a private system, a\ncredential nobody can supply, a capability that needs a human. It is not the\nanswer to "the registry had nothing useful".\n\nTool node (one direct call, no LLM — the right shape for "read this file"):\n  {"id": "load_file", "kind": "tool", "tools": ["read_file"],\n   "tool_args": {"path": "{file_path}"}, "writes": "file_text"}\n\n# Control-flow cookbook (add_node `node` argument — copy these shapes)\nWHEN to use what: different handling per category → `router`; retry/refine until\ngood → `loop`; same processing for every item of a list → `map`; a step that only\nsometimes applies → a `when:` guard; a risky step needing a fallback → `on_error`.\nA plain linear pipeline needs NONE of these — don\'t force control flow.\nIf the plan asked for one of these, build that shape — do not flatten it.\n\nRouter (the router ITSELF classifies — no separate classify node needed; every\ntarget depends_on the router):\n  {"id": "route", "kind": "router",\n   "goal": "Route this support ticket by urgency: {ticket}",\n   "routes": {"urgent": "escalate", "routine": "archive"},\n   "default": "archive"}\n(Advanced: deterministic routing on a prior node\'s output uses\n "cases": [{"when": "contains(lower(nodes.check), \'yes\')", "to": "…"}] instead.)\n\nLoop (iterate a nested body until good; state the stop condition in PLAIN ENGLISH\nvia `until` — an internal judge decides stop/continue each iteration and its\nreason reaches the next attempt as {feedback}):\n  {"id": "refine", "kind": "loop", "max_iterations": 3,\n   "until": "the review approves the draft",\n   "body": [{"id": "draft", "kind": "base",\n             "goal": "Draft it. Reviewer feedback from last attempt: {feedback}"},\n            {"id": "review", "kind": "base", "depends_on": ["draft"],\n             "goal": "Review the draft critically."}]}\n(`until` is the only stop condition. For a check that code can make — budgets,\n cursors, counts, thresholds — write it as a function in the graph\'s\n `functions:` file and set "until": "<function_name>": it is free and exact,\n where plain English costs one LLM call per iteration. Plain English is for\n judgements only a reader can make. Either way it must describe what the body\n actually produces: a condition about a different subject stops the loop.)\n\nMap (fan out over a list; the node\'s output is the ordered per-item results):\n  {"id": "per_item", "kind": "map", "over": "inputs.items", "as": "item",\n   "body": [{"id": "handle", "kind": "base",\n             "purpose": "Process one item: {item}", "goal": "…"}]}\n\n# Your capabilities (auto-derived, version-pinned)\n# Neurosurfer capabilities (v222ceee9132d, neurosurfer 1.0.0)\n\n## Node kinds\n- **base** — One LLM call, for writing/summarising/classifying/transforming text. It CANNOT take tools — it only sees its prompt. A step that must touch the outside world is a `tool` or `react` node.\n    - requires: output_schema (output schema)\n    - requires: Say what the step should do in `instructions`. The older purpose/goal/expected_result trio is still read when it is absent.\n    - requires: For structured output set an output schema — without one, a node asked for an object returns JSON as a string.\n    - requires: Tools are optional here and the model gets one round with them. A step that must call tools repeatedly to finish its job is a `react` node.\n    - note: Structured output is `mode: structured` + `output_schema` (an import path to a pydantic model). Without it a base node returns text, so a node asked for an object emits JSON *as a string* and anything checking the shape fails. A build blocked over exactly this, reporting that structured output \'is not available with the current node schema\' — it is, and nothing had told it.\n- **function** — Deterministic Python: imports and calls `callable` with inputs + dep outputs.\n    - requires: callable (function to call)\n    - requires: Arguments are matched to the signature **by name**, so a parameter whose name is not a graph input or an upstream node id receives nothing. Nothing warns about this today — see plan 01, phase 3.\n- **input** — Human-in-the-loop pause. Resolves from a pre-supplied input/var named by `writes` (or the node id) — the API resume path — else asks interactively; otherwise the run finishes as awaiting_input.\n    - requires: In `dict` mode the fields a person fills in are the workflow\'s declared inputs — add them on this node.\n    - requires: With nothing supplied the run finishes `awaiting_input`, not `failed`; a client resumes it. Resuming currently re-runs the whole graph, so a node before this one runs twice.\n- **loop** — Repeats a nested `body` sub-graph until a stop condition holds. A CONTINUE verdict from `until` reaches the next iteration as {feedback}; `accumulate` collects each output into a list.\n    - requires: body (body)\n    - requires: max_iterations (maximum iterations)\n    - requires: No `until` means it runs to the ceiling.\n    - requires: A plain-English `until` costs one LLM call per iteration; a function costs nothing. Prefer a function whenever the condition is checkable in code.\n    - requires: A plain-English `until` that is about a different subject than the body produces stops the loop with a warning rather than running to the ceiling — so the condition must actually describe the body\'s output.\n- **map** — Runs a nested `body` once per item of the collection from the `over` expression, `concurrency` at a time. Output is the ordered list of per-item results (implicit gather).\n    - requires: body (body)\n    - requires: over (over)\n    - requires: The output is the ordered list of per-item results — the gather is implicit, so no node is needed to collect them.\n- **output** — What the graph returns. With `value` set it is an interpolated template over graph inputs, upstream outputs and `writes` vars; without one it passes its single dependency through unchanged, preserving its type.\n    - requires: Needs something to return: a dependency whose output it passes through, or a value template. With neither it returns nothing.\n    - requires: Nothing may depend on one, and it cannot be an error target.\n    - requires: An unresolved placeholder in the value is refused, not warned about — this text is what a caller receives.\n    - requires: Takes precedence over the older graph-level `outputs:` list.\n- **python** — Alias of function (imports and calls a Python callable).\n    - requires: callable (function to call)\n    - requires: Currently identical to `function` — same executor path, same required import path. It does not run inline code.\n- **react** — An LLM that calls tools, for any step that must touch the outside world AND needs a model to decide what to send. That covers multi-step filesystem/search/web/shell work, and also the single call whose arguments must be composed — the SQL for a query tool, the phrase for a search tool, the body for an API tool.\n    - requires: tools (tools)\n    - requires: output_schema (output schema)\n    - requires: The only kind that both reasons and acts. `base` reasons and cannot act; `tool` acts and cannot reason.\n    - note: This is the ONLY kind that both reasons and acts. `base` reasons and cannot act; `tool` acts and cannot reason. A step needing both is a react node — splitting it into a base node that writes an instruction and a tool node that was supposed to read it does not work, because nothing passes the instruction to the tool.\n    - note: `tool_args` here are BOUND arguments, not the whole call: the engine supplies them on every call the model makes and removes them from the schema it is offered. This is how a react node uses a credential — `secrets: [DB_URL]` with `tool_args: {dsn: \'${DB_URL}\'}` means the model composes only the query, and never sees the connection string.\n- **router** — Selects ONE downstream branch; non-selected targets are pruned. `routes` ({label: target}) classifies with one LLM call; `cases` ([{when, to}]) evaluates predicates and costs nothing.\n    - requires: Declare `routes` or `cases`, never both — routes classify with a model, cases evaluate expressions.\n    - requires: Every target must declare this router in its `depends_on`.\n    - requires: The `cases` form makes no LLM call, so the provider here means nothing for it.\n- **subgraph** — Runs a nested `body` sub-graph once (composition). Its final outputs become this node\'s output.\n    - requires: body (body)\n    - requires: Body nodes may only depend on their siblings — a dependency pointing outside the body is refused when the graph loads.\n- **tool** — Directly invokes ONE registered tool. Use it when every required argument is a constant or an interpolation of an input/upstream output (`read_file` with `tool_args: {path: \'{doc}\'}`).\n    - requires: tools (tool)\n    - requires: There is no LLM call, so nothing composes its arguments and nothing reads an instruction. If an argument has to be *worked out* — a query, a search phrase, a request body — the step is a `react` node with that tool attached.\n    - requires: tool_args supplying every required parameter of that tool — from an argument here, or a graph input or upstream output of the same name. One missing calls the tool with nothing and fails at run time.\n    - requires: A credential goes in `secrets` and is written `${NAME}` in an argument, never in a prompt.\n    - note: A stored value (database password, connection string, API key) is reached by naming it in `secrets: [NAME]` and writing `${NAME}` in `tool_args`. It is filled at call time and never enters a prompt.\n    - note: NEVER write ${NAME} into purpose/goal/expected_result — those reach the model, and validation refuses it. If a step needs a credential, it is a `tool` node, not a `base` one.\n\n## Build rules (each one earned by a build that failed)\n- **A `tool` node has NO model. If you cannot fill `tool_args` completely right now, the step is a `react` node with that tool attached.**\n    - why: A tool node\'s goal text is read by nobody — there is no model call in it. Writing an instruction there and expecting the tool to follow it produces a call with missing arguments.\n- **A credential is NEVER a graph input. Name it in `secrets: [NAME]` on the node that needs it and write `${NAME}` inside `tool_args`.**\n    - why: Graph inputs are in the interpolation scope, so an input named `db_password` is reachable from every node\'s prompt — and from there the model\'s context, the trace, and any exporter. Secrets live outside that scope entirely and reach only the tool call.\n- **Check the tool can do the thing, not just that its name matches. If nothing in the catalog can, say so — do not attach the closest name.**\n    - why: A node that validates with a plausible-but-wrong tool is worse than a blocked build: it goes green and produces a file that is the wrong format, or an empty result presented as an answer.\n- **A step that reads or writes anything outside the model is external, however ordinary it sounds. \'Query the orders table\' is external; so is \'save the report\'.**\n    - why: Steps marked internal never reach the capability ladder, so nothing checks whether a tool for them exists. The error surfaces much later, as a node that cannot run.\n- **Every input the workflow declares must be named by a step — as `{name}` in a goal, in an `over`/`when` expression, in `tool_args`, or as a parameter of a code node\'s function. If no step needs it, do not declare it.**\n    - why: A node\'s turn is its own task text plus the outputs of its `depends_on`, and nothing else. An input no step names is therefore read by nobody: the caller passes it, the run goes green, and the answer is written as though it had never been supplied. Validation warns but does not refuse, so this ships.\n- **Finish by calling a terminal tool. A graph that validates is not a build that ended.**\n    - why: Stopping after the last node leaves the build hanging until a nudge restarts it, and each nudge spends turns that the rest of the build then does not have.\n\n## Worked shapes\n- every argument known → a tool node, no model:\n  `{"id": "read_doc", "kind": "tool", "tools": ["read_file"], "tool_args": {"path": "{doc_path}"}, "writes": "doc"}`\n- an argument must be composed → a react node with the tool:\n  `{"id": "research", "kind": "react", "tools": ["web_search"], "goal": "Search for recent, credible sources about {topic}. Return the three most useful with a one-line note on what each contributes.", "writes": "sources"}`\n- a credential reaches the tool and never a prompt:\n  `{"id": "fetch_items", "kind": "tool", "tools": ["http"], "secrets": ["API_TOKEN"], "tool_args": {"url": "https://api.example.com/v1/items", "headers": {"Authorization": "Bearer ${API_TOKEN}"}}, "writes": "items"}`\n\n## Expressions (guards, router cases, over)\n- functions: abs, all, any, bool, contains, endswith, float, int, len, lower, max, min, round, sorted, startswith, str, sum, upper\n- namespaces: inputs, nodes, vars, state, index/item (loop & map scope)\n- Predicates evaluate against namespaces: inputs.*, nodes.<id> (a node\'s raw output), vars.* (explicit `writes`), plus index/item inside loop/map bodies. Real LLM output arrives with whitespace/case noise — prefer contains(lower(nodes.x), \'label\') over exact equality, or use structured outputs for exact matching. Missing keys resolve to None (predicates fail closed, they never crash the run).\n\n## Workflow-node tools (the ONLY tools you may assign to nodes)\n- `apply_edit` — Replace exact string(s) in a file. Fails if an old_string is missing or ambiguous (unless replace_all). Pass `edits` (a list of old_string/new_string hunks) to   [file.edit]\n- `browse` — Open a URL in a headless browser (renders JavaScript) and return the page\'s readable text. Use this instead of `http` when a page needs JS to show its content.   [web.browse]\n- `data` — Inspect or query a local structured-data file. CSV/TSV → columns + preview rows; JSON → structure, or a value at a dotted key-path; JSONL → record preview; SQLi  [data.inspect]\n- `http` — Make an HTTP request to a URL and return the status, key response headers, and body (JSON pretty-printed; long text truncated). Use it to call REST/JSON APIs or  [web.request]\n- `list_dir` — List directory entries, or glob a pattern under a directory (e.g. pattern=\'**/*.py\'). Skips VCS/build noise and .gitignore\'d paths.  [file.list]\n- `read_file` — Read a file and return its contents. Text files come back with line numbers (use offset/limit for large files). Image files (png, jpg/jpeg, gif, webp) are retur  [file.read]\n- `run_command` — Run a shell command in the working directory (or `cwd`, if given) and return its combined stdout/stderr and exit code. Subject to the Task\'s shell policy gate.   [system.shell]\n- `search` — Search file contents by regular expression (ripgrep-style). Returns file:line: matched-line. Filter files with the glob argument.  [file.search]\n- `sql` — Read-only access to a SQL database server (SQL Server, PostgreSQL, MySQL and anything else SQLAlchemy speaks). Operations: `test_connection`, `list_tables`, `ta  [db.connect, db.query, db.schema]  needs dsn as ${SECRET}\n- `web_search` — Search the web for current information. Supports DuckDuckGo (free, default) and SerpAPI/Google (requires SERPAPI_API_KEY). Returns result titles, URLs and snipp  [web.search]  needs api_key as ${SECRET}\n- `write_file` — Create or overwrite a whole file with the given contents, creating parent directories as needed. Prefer apply_edit to modify an existing file — it changes only   [file.write]\n\n## Capabilities NOTHING here provides\n- chart.render, inbox.read, message.send, pdf.render\n  A step needing one of these cannot be built with a catalog tool. Author one, import an MCP server for it, or declare_blocked — never substitute a tool that merely sounds close.\n\n## Execution API\n- GET /v1/runs/{run_id}/events is SSE (replay + live tail)\n- DELETE /v1/runs/{run_id}; DELETE /v1/workflows/{name}; GET /v1/runs; GET /v1/runs/{run_id}; GET /v1/runs/{run_id}/events; GET /v1/runs/{run_id}/nodes/{node_id}; GET /v1/runs/{run_id}/trace; GET /v1/workflows; GET /v1/workflows/{name}; GET /v1/workflows/{name}/requirements; POST /v1/runs/{run_id}/resume; POST /v1/workflows; POST /v1/workflows/validate; POST /v1/workflows/{name}/runs; PUT /v1/workflows/{name}\n'

""")


'You are the Neurosurfer Architect — an agent that designs and builds runnable
workflow packages from a user's plain-English intent.

# Operating procedure
1. BUILD. `set_workflow` first (name, description, declared inputs), then
   `add_node` one node at a time — read every warning and fix it.
   - **With a plan** (usually — it is in your first message): build it. One node
     per step, keeping the step ids. The tools are already chosen for you; use
     exactly those. If a step's id differs from your node's, pass `plan_step_id`.
     Every planned step must be built or explicitly `drop_plan_step`ped.
   - **Without a plan**: design as you go. One node per distinct capability or
     decision; let the work set the size, not a target count. Map each node to a
     clause the user actually asked for — do NOT add validation, formatting or
     "nice-to-have" steps they did not request.
2. GROUND anything the plan didn't. For each node ask: *does this reach outside
   the model?* Readin